# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring**, same March 2026 slice as ML-04 and ML-07.

ML-07 built a rule and checked the two beliefs it rests on. This is the deeper version: shapes
first, then three beliefs I have **not** tested yet, then the flag-linked question ML-07 left open —
**why do pages at position 1–2 have the lowest CTR of any shallow band?**

Two rules held throughout. **No verdict from a bucket under 50 rows** (30 for cross-cuts) — a big
ratio from a small cell is noise wearing a costume, and "insufficient data" is a finding too. And
**every rate is pooled** (total clicks ÷ total impressions), never an average of per-page rates.

Verdict words exactly as the session defined them: **CONFIRMED · OPPOSITE · MIXED · FALSE.**
A negative verdict is a win — it stops a rule before it ships.

## 0. Setup — the same slice, rebuilt

Identical to ML-07 so this notebook runs standalone: March 2026, one row per page, positions
`+1`-corrected for the zero-based convention found in ML-04.

In [12]:
%pip -q install duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [13]:
import duckdb, pandas as pd, numpy as np
from scipy.stats import spearmanr
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 50)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL, MONTH = "hf://datasets/FlyRank/internship-warehouse", "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0
FLOOR, CROSS_FLOOR = 50, 30      # sample-size floors: no verdict below these

pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                                  AS impressions_31d,
           SUM(gsc_clicks)                                                       AS clicks_31d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)    AS days_with_impressions_31d,
           MAX(gsc_impressions)                                                  AS top_day_impressions,
           SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_num,
           SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_den,
           STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS position_volatility_31d,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-18')  AS imp_last14,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-04'
                                          AND report_date <  DATE '2026-03-18')  AS imp_prev14
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

pages["avg_position_31d"] = pages.pos_num / pages.pos_den.replace(0, np.nan) + 1
pages["ctr_pp"]           = 100 * pages.clicks_31d / pages.impressions_31d.replace(0, np.nan)

lane = pages[(pages.impressions_31d >= MIN_IMPRESSIONS) &
             (pages.days_with_impressions_31d >= MIN_ACTIVE_DAYS) &
             (pages.avg_position_31d >= MIN_POSITION)].copy()
lane["top_day_impression_share"] = lane.top_day_impressions / lane.impressions_31d
lane["momentum_log14v14"]        = np.log((lane.imp_last14.fillna(0) + 1) /
                                          (lane.imp_prev14.fillna(0) + 1))
print(f"all pages with GSC data: {len(pages):,}   |   eligible lane slice: {len(lane):,} pages, "
      f"{lane.client_hash_id.nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

all pages with GSC data: 176,738   |   eligible lane slice: 61,881 pages, 36 clients


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before any correlation, look at the shape. Web traffic is almost always **heavy-tailed** — a few
giants and a very long tail of small pages — and that single fact decides which statistics are
allowed downstream.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

KEY = ["impressions_31d", "clicks_31d", "ctr_pp", "avg_position_31d",
       "days_with_impressions_31d", "position_volatility_31d"]
print("DISTRIBUTIONS of the fields this audit uses (eligible lane slice)")
display(lane[KEY].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(3))

print("\nHeavy-tail evidence:")
for c in ["impressions_31d", "clicks_31d"]:
    s = lane[c]
    top1 = s.nlargest(max(1, int(len(s) * 0.01))).sum() / max(s.sum(), 1)
    print(f"  {c:18} mean {s.mean():>10,.0f} vs median {s.median():>8,.0f} "
          f"({s.mean()/max(s.median(),1):.1f}x) | top 1% of pages hold {top1:.1%} of the total")
print("\nOBSERVED: the mean sits well above the median for both, and a hundredth of the pages "
      "carries a large share of all traffic. That is the textbook heavy tail -- and it means a "
      "plain average over pages is a statement about the giants, not about a typical page.")

DISTRIBUTIONS of the fields this audit uses (eligible lane slice)


,impressions_31d,clicks_31d,ctr_pp,avg_position_31d,days_with_impressions_31d,position_volatility_31d
count,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000
mean,4345.171,12.810,0.283,12.443,29.533,4.494
std,8516.190,43.998,0.353,11.287,3.831,4.014
min,500.000,0.000,0.000,1.046,5.000,0.131
5%,565.000,0.000,0.000,3.062,20.000,0.668
25%,941.000,1.000,0.064,5.138,31.000,1.523
50%,1900.000,4.000,0.178,7.786,31.000,3.373
75%,4548.000,12.000,0.384,15.864,31.000,6.133
95%,15691.000,49.000,0.913,36.179,31.000,12.256
99%,37547.800,141.000,1.657,52.113,31.000,18.851



Heavy-tail evidence:
  impressions_31d    mean      4,345 vs median    1,900 (2.3x) | top 1% of pages hold 14.2% of the total
  clicks_31d         mean         13 vs median        4 (3.2x) | top 1% of pages hold 21.8% of the total

OBSERVED: the mean sits well above the median for both, and a hundredth of the pages carries a large share of all traffic. That is the textbook heavy tail -- and it means a plain average over pages is a statement about the giants, not about a typical page.


### What the tail does to a correlation

The skill's warning, tested rather than repeated: on heavy-tailed columns a plain Pearson
correlation is dominated by the largest rows. Same relationship — *do pages with more impressions
get more clicks?* — measured four ways.

In [15]:
x, y = lane.impressions_31d, lane.clicks_31d
pear_raw = np.corrcoef(x, y)[0, 1]
pear_log = np.corrcoef(np.log1p(x), np.log1p(y))[0, 1]
spear    = spearmanr(x, y)[0]

small = lane[lane.impressions_31d <= lane.impressions_31d.quantile(0.99)]
pear_no_giants = np.corrcoef(small.impressions_31d, small.clicks_31d)[0, 1]

print(f"impressions vs clicks, {len(lane):,} pages")
print(f"  Pearson, raw values         : {pear_raw:.3f}")
print(f"  Pearson, after log1p        : {pear_log:.3f}")
print(f"  Spearman (rank-based)       : {spear:.3f}")
print(f"  Pearson raw, top 1% removed : {pear_no_giants:.3f}   (n={len(small):,})")
print(f"\nOBSERVED: the measures disagree by {abs(pear_raw - spear):.2f}, and removing the largest "
      f"1% of pages moves the raw Pearson by {abs(pear_raw - pear_no_giants):.2f}.")
print("CONSEQUENCE, applied for the rest of this notebook: every test below uses bucket tables and "
      "POOLED rates rather than correlations on raw values -- on this shape, a single coefficient "
      "describes the tail and hides the population.")

impressions vs clicks, 61,881 pages
  Pearson, raw values         : 0.690
  Pearson, after log1p        : 0.713
  Spearman (rank-based)       : 0.681
  Pearson raw, top 1% removed : 0.602   (n=61,262)

OBSERVED: the measures disagree by 0.01, and removing the largest 1% of pages moves the raw Pearson by 0.09.
CONSEQUENCE, applied for the rest of this notebook: every test below uses bucket tables and POOLED rates rather than correlations on raw values -- on this shape, a single coefficient describes the tail and hides the population.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three beliefs my lane quietly assumes and has not tested. Same structure each time: the claim in one
sentence, a bucket table with **n**, one verdict word, and what it changes.

None are label-derived — they use volume, stability, trend and client identity, all knowable at the
decision moment.

### Signal 1 — "normal CTR" means the same thing for every client

My proxy compares every page to a **global** position-tier baseline. That is only fair if a normal
CTR at position 5 means roughly the same for every client. ML-04 hinted otherwise: the model got the
*ranking* roughly right on unseen clients but the *magnitude* worse than guessing the mean — the
signature of a per-client level shift. This tests it head-on.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

cl = lane.groupby("client_hash_id").agg(
        n_pages=("ctr_pp", "size"), impressions=("impressions_31d", "sum"),
        clicks=("clicks_31d", "sum"), median_position=("avg_position_31d", "median"))
cl["pooled_ctr_pp"] = (100 * cl.clicks / cl.impressions).round(3)
cl = cl[cl.n_pages >= FLOOR].sort_values("impressions", ascending=False)

print(f"SIGNAL 1 - pooled CTR per client (clients with >= {FLOOR} pages: {len(cl)} of "
      f"{lane.client_hash_id.nunique()})")
display(cl[["n_pages", "impressions", "pooled_ctr_pp", "median_position"]].head(10).round(3))

lo, hi   = cl.pooled_ctr_pp.min(), cl.pooled_ctr_pp.max()
spread   = hi / max(lo, 1e-9)
lane_pp  = 100 * lane.clicks_31d.sum() / lane.impressions_31d.sum()
print(f"\nacross {len(cl)} clients: lowest {lo:.3f}% | median {cl.pooled_ctr_pp.median():.3f}% | "
      f"highest {hi:.3f}%  ->  {spread:.0f}x spread")
print(f"the single global baseline my rule compares against is {lane_pp:.3f}%")
print(f"clients whose own pooled CTR is more than 2x away from that global number: "
      f"{int(((cl.pooled_ctr_pp > 2*lane_pp) | (cl.pooled_ctr_pp < lane_pp/2)).sum())} of {len(cl)}")

verdict_1 = ("CONFIRMED" if spread < 1.5 else "MIXED" if spread < 3 else "FALSE")
print(f"\nVERDICT 1: {verdict_1}  (claim tested: 'normal CTR is the same across clients')")

SIGNAL 1 - pooled CTR per client (clients with >= 50 pages: 23 of 36)


,n_pages,impressions,pooled_ctr_pp,median_position
client_hash_id,,,,
client_73cda7b4e4f265ea,14246,71481813.0,0.313,5.799
client_23a62021009f63c4,9999,56080643.0,0.226,21.824
client_62f4a7e64f5e0096,11072,51337322.0,0.246,5.584
client_e547b89c05043229,6058,22848591.0,0.326,9.402
client_20259bd6705d81d4,3024,17736202.0,0.365,14.307
client_fef1a8f436438636,5245,15531817.0,0.289,11.813
client_e5c2aa26a8598242,2216,9425830.0,0.438,9.109
client_08a6a72ff48e62c0,2658,9046953.0,0.439,7.359
client_a80fca3f171ed1de,1306,3808074.0,0.207,5.573



across 23 clients: lowest 0.115% | median 0.289% | highest 1.159%  ->  10x spread
the single global baseline my rule compares against is 0.295%
clients whose own pooled CTR is more than 2x away from that global number: 6 of 23

VERDICT 1: FALSE  (claim tested: 'normal CTR is the same across clients')


**What verdict 1 changes.** If the spread is wide, a page can be flagged for sitting below the
*global* tier baseline while sitting perfectly normally inside its own client — and a genuinely weak
page at a high-CTR client can hide. That makes per-client normalisation a measured requirement for
ML-05 rather than an inference from a negative R².

### Signal 2 — spiky traffic makes a monthly CTR unrepresentative

The belief behind ML-07's `one-day spike` caveat: if most of a page's month arrived on one day, its
monthly CTR describes an unusual day rather than the page. Never tested — just asserted.

In [17]:
SB = [0, 0.10, 0.20, 0.30, 0.50, 1.01]
SL = ["<10%", "10-20%", "20-30%", "30-50%", "50%+"]
lane["spike_band"] = pd.cut(lane.top_day_impression_share, bins=SB, labels=SL)

g2 = lane.groupby("spike_band", observed=True)
s2 = pd.DataFrame({
    "n_pages":          g2.size(),
    "impressions":      g2.impressions_31d.sum(),
    "pooled_ctr_pp":   (100 * g2.clicks_31d.sum() / g2.impressions_31d.sum()).round(3),
    "share_zero_click": g2.clicks_31d.apply(lambda s: (s == 0).mean()).round(3),
    "median_days_live": g2.days_with_impressions_31d.median(),
})
print("SIGNAL 2 - does a one-day spike change what CTR looks like?")
display(s2)

usable2 = s2[s2.n_pages >= FLOOR]
if len(usable2) < 2:
    verdict_2 = "INSUFFICIENT DATA"
else:
    falls   = usable2.pooled_ctr_pp.iloc[0] > usable2.pooled_ctr_pp.iloc[-1]
    zc_rise = usable2.share_zero_click.iloc[-1] > usable2.share_zero_click.iloc[0]
    verdict_2 = ("CONFIRMED" if falls and zc_rise else
                 "MIXED"     if falls or zc_rise  else "FALSE")
    print(f"\nbuckets usable (n >= {FLOOR}): {len(usable2)} of {len(s2)}")
    print(f"steadiest {usable2.pooled_ctr_pp.iloc[0]:.3f}% CTR / "
          f"{usable2.share_zero_click.iloc[0]:.1%} zero-click   vs   spikiest "
          f"{usable2.pooled_ctr_pp.iloc[-1]:.3f}% / {usable2.share_zero_click.iloc[-1]:.1%}")
print(f"\nVERDICT 2: {verdict_2}  (claim tested: 'spiky pages have unrepresentative CTR')")

SIGNAL 2 - does a one-day spike change what CTR looks like?


,n_pages,impressions,pooled_ctr_pp,share_zero_click,median_days_live
spike_band,,,,,
<10%,51102,239771321.0,0.298,0.154,31.0
10-20%,8972,23758534.0,0.275,0.254,31.0
20-30%,1129,3170807.0,0.206,0.326,31.0
30-50%,512,1666257.0,0.219,0.365,30.0
50%+,166,516577.0,0.314,0.554,29.0



buckets usable (n >= 50): 5 of 5
steadiest 0.298% CTR / 15.4% zero-click   vs   spikiest 0.314% / 55.4%

VERDICT 2: MIXED  (claim tested: 'spiky pages have unrepresentative CTR')


### Signal 3 — pages losing impressions also convert worse

The belief behind ML-07's `demand falling` caveat, and behind refresh logic generally: a sliding
page is in trouble on more than one axis. If CTR is flat across the momentum range, then falling
impressions is a **separate disease** from a weak snippet — and the session's whole point was that
the wrong medicine wastes a month.

In [18]:
MB  = [-np.inf, -1, -0.3, 0.3, 1, np.inf]
ML_ = ["falling hard", "falling", "flat", "rising", "rising hard"]
lane["momentum_band"] = pd.cut(lane.momentum_log14v14, bins=MB, labels=ML_)

g3 = lane.groupby("momentum_band", observed=True)
s3 = pd.DataFrame({
    "n_pages":          g3.size(),
    "impressions":      g3.impressions_31d.sum(),
    "pooled_ctr_pp":   (100 * g3.clicks_31d.sum() / g3.impressions_31d.sum()).round(3),
    "median_position":  g3.avg_position_31d.median().round(2),
    "share_zero_click": g3.clicks_31d.apply(lambda s: (s == 0).mean()).round(3),
})
print("SIGNAL 3 - do pages losing impressions also have worse CTR?")
display(s3)

usable3 = s3[s3.n_pages >= FLOOR]
if len(usable3) < 2:
    worst = best = ratio3 = float("nan"); verdict_3 = "INSUFFICIENT DATA"
else:
    worst, best = usable3.pooled_ctr_pp.iloc[0], usable3.pooled_ctr_pp.max()
    ratio3 = best / max(worst, 1e-9)
    verdict_3 = ("CONFIRMED" if ratio3 > 1.5 and worst == usable3.pooled_ctr_pp.min() else
                 "MIXED"     if ratio3 > 1.2 else "FALSE")
    print(f"\nbuckets usable (n >= {FLOOR}): {len(usable3)} of {len(s3)}")
    print(f"falling-hard {worst:.3f}% vs best bucket {best:.3f}%  ->  {ratio3:.2f}x")
print(f"\nVERDICT 3: {verdict_3}  (claim tested: 'falling impressions means worse CTR')")
print("\nEither answer is useful. A weak result says demand and snippet are separate problems, and "
      "my rule must not treat one as the other; a strong one says momentum belongs in the score.")

SIGNAL 3 - do pages losing impressions also have worse CTR?


,n_pages,impressions,pooled_ctr_pp,median_position,share_zero_click
momentum_band,,,,,
falling hard,2571,8483054.0,0.169,8.14,0.298
falling,11657,46742649.0,0.192,9.72,0.216
flat,28199,124998790.0,0.297,7.82,0.157
rising,13498,69125580.0,0.357,7.18,0.149
rising hard,5956,19533423.0,0.358,7.36,0.176



buckets usable (n >= 50): 5 of 5
falling-hard 0.169% vs best bucket 0.358%  ->  2.12x

VERDICT 3: CONFIRMED  (claim tested: 'falling impressions means worse CTR')

Either answer is useful. A weak result says demand and snippet are separate problems, and my rule must not treat one as the other; a strong one says momentum belongs in the score.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's CTR-fix flag rests on one assumption: **pages at a similar position are a fair comparison
group.** ML-07 tested it and found the belief holds from position 3 downward — but breaks at the
very top. Pages ranking 1–2 pooled to **0.118% CTR**, the lowest of any shallow band, with 35%
taking zero clicks all month. The assumption fails exactly where my queue is most confident.

So: is it real, or is it a few rows? Three candidates, each with a different consequence:

| if… | then the rule should… |
|---|---|
| **one client dominates the band** | compare within client, not globally (signal 1's fix) |
| **a few giant pages dominate** | treat it as outliers, not a pattern |
| **many clients show it independently** | it is a property of the top positions — escalate it |

The test that separates them: take clients appearing in **both** the 1–2 band and the 3–5 band, and
compare each client against *itself*.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

POS_BANDS  = [0, 2, 3, 5, 10, 20, 50, np.inf]
POS_LABELS = ["1.0-2", "2-3", "3-5", "5-10", "10-20", "20-50", "50+"]
lane["pos_band"] = pd.cut(lane.avg_position_31d, bins=POS_BANDS, labels=POS_LABELS)

band12 = lane[lane.pos_band == "1.0-2"]
band35 = lane[lane.pos_band == "3-5"]
band12_pp = 100 * band12.clicks_31d.sum() / band12.impressions_31d.sum()
band35_pp = 100 * band35.clicks_31d.sum() / band35.impressions_31d.sum()
print(f"band 1-2: {len(band12):>6,} pages, {band12.impressions_31d.sum():>12,.0f} impressions, "
      f"pooled CTR {band12_pp:.3f}%")
print(f"band 3-5: {len(band35):>6,} pages, {band35.impressions_31d.sum():>12,.0f} impressions, "
      f"pooled CTR {band35_pp:.3f}%")

# --- candidate A: one client carrying the band? ----------------------------
by_client = band12.groupby("client_hash_id").agg(
    n_pages=("ctr_pp", "size"), impressions=("impressions_31d", "sum"),
    clicks=("clicks_31d", "sum")).sort_values("impressions", ascending=False)
by_client["share_of_band"] = (by_client.impressions / by_client.impressions.sum()).round(3)
print(f"\nA) clients in the 1-2 band: {len(by_client)}; the largest holds "
      f"{by_client.share_of_band.iloc[0]:.1%} of its impressions")
drop1 = by_client.iloc[1:]
ctr_wo_client = 100 * drop1.clicks.sum() / max(drop1.impressions.sum(), 1)
print(f"   pooled CTR with that client removed : {ctr_wo_client:.3f}%  (vs {band12_pp:.3f}%)")

# --- candidate B: a few giant pages? ---------------------------------------
no_giants = band12[band12.impressions_31d <= band12.impressions_31d.quantile(0.90)]
ctr_wo_giants = 100 * no_giants.clicks_31d.sum() / max(no_giants.impressions_31d.sum(), 1)
print(f"B) pooled CTR with the top 10% largest PAGES removed: {ctr_wo_giants:.3f}%  "
      f"(n={len(no_giants):,})")

band 1-2:    483 pages,    1,991,779 impressions, pooled CTR 0.118%
band 3-5: 11,631 pages,   68,052,893 impressions, pooled CTR 0.394%

A) clients in the 1-2 band: 14; the largest holds 30.6% of its impressions
   pooled CTR with that client removed : 0.090%  (vs 0.118%)
B) pooled CTR with the top 10% largest PAGES removed: 0.149%  (n=434)


In [20]:
# --- candidate C: do individual clients show it against THEMSELVES? --------
a = band12.groupby("client_hash_id").agg(n12=("ctr_pp", "size"),
                                         imp12=("impressions_31d", "sum"),
                                         clk12=("clicks_31d", "sum"))
b = band35.groupby("client_hash_id").agg(n35=("ctr_pp", "size"),
                                         imp35=("impressions_31d", "sum"),
                                         clk35=("clicks_31d", "sum"))
joined = a.join(b, how="inner")

# The 1-2 band is small, so the preferred cross-cut floor may leave nothing. Try it, then fall
# back once to a lower floor -- and SAY which one produced the table rather than quietly relaxing.
used_floor = CROSS_FLOOR
both = joined[(joined.n12 >= CROSS_FLOOR) & (joined.n35 >= CROSS_FLOOR)]
if len(both) < 3:
    used_floor = 10
    both = joined[(joined.n12 >= used_floor) & (joined.n35 >= used_floor)]
    print(f"C) the preferred cross-cut floor of {CROSS_FLOOR} pages left only "
          f"{int(((joined.n12 >= CROSS_FLOOR) & (joined.n35 >= CROSS_FLOOR)).sum())} clients -- "
          f"falling back to {used_floor}, and the result below is DIRECTIONAL, not a verdict I "
          f"would defend on its own.")

both = both.copy()
both["ctr_12_pp"] = (100 * both.clk12 / both.imp12).round(3)
both["ctr_35_pp"] = (100 * both.clk35 / both.imp35).round(3)
both["ratio_35_over_12"] = (both.ctr_35_pp / both.ctr_12_pp.replace(0, np.nan)).round(2)

print(f"C) clients with >= {used_floor} pages in BOTH bands: {len(both)}")
if len(both) >= 3:
    display(both[["n12", "ctr_12_pp", "n35", "ctr_35_pp", "ratio_35_over_12"]]
            .sort_values("ratio_35_over_12", ascending=False))
    worse = int((both.ctr_12_pp < both.ctr_35_pp).sum())
    print(f"\nclients whose OWN 1-2 pages convert worse than their OWN 3-5 pages: "
          f"{worse} of {len(both)}")
    verdict_flag = ("CONFIRMED" if worse >= 0.7 * len(both) else
                    "MIXED"     if worse >= 0.4 * len(both) else "FALSE")
    if used_floor < CROSS_FLOOR:
        verdict_flag += " (below preferred floor)"
else:
    worse, verdict_flag = None, "INSUFFICIENT DATA"
    print("   fewer than 3 clients clear the cross-cut floor -- no verdict from cells this small.")
    print("   That is itself a finding: the 1-2 band is thin almost everywhere, which is why the")
    print("   whole-band number was fragile in the first place.")

print(f"\nVERDICT (flag-linked): {verdict_flag}")
print("Claim tested: 'pages at a similar position are a fair comparison group'. Wherever individual "
      "clients reproduce the dip against their own higher-position pages, it is a property of the "
      "top positions rather than of one client or a few giants -- and a rule that treats a 1-2 page "
      "like any other is comparing it to an expectation it structurally cannot meet.")

C) clients with >= 30 pages in BOTH bands: 5


,n12,ctr_12_pp,n35,ctr_35_pp,ratio_35_over_12
client_hash_id,,,,,
client_23a62021009f63c4,36,0.018,363,0.328,18.22
client_e547b89c05043229,47,0.072,843,0.456,6.33
client_73cda7b4e4f265ea,112,0.114,4346,0.407,3.57
client_a80fca3f171ed1de,75,0.097,359,0.261,2.69
client_62f4a7e64f5e0096,151,0.180,3486,0.314,1.74



clients whose OWN 1-2 pages convert worse than their OWN 3-5 pages: 5 of 5

VERDICT (flag-linked): CONFIRMED
Claim tested: 'pages at a similar position are a fair comparison group'. Wherever individual clients reproduce the dip against their own higher-position pages, it is a property of the top positions rather than of one client or a few giants -- and a rule that treats a 1-2 page like any other is comparing it to an expectation it structurally cannot meet.


### What I do about it — before the model, not after

Whatever the verdict, the response has the same shape: **the rule does not change, the confidence
attached to it does.** ML-07 already downgrades every position ≤3 pick and lists them as weak picks.
This audit gives that downgrade a reason with a number behind it.

Two explanations I cannot separate with this table, and will not pretend to:

- **Zero-click SERPs.** A page ranking 1–2 for something Google answers directly in the results — a
  definition, a date, a conversion — earns the impression and never the click. That would make the
  low CTR *correct behaviour*, and no rewritten snippet would touch it.
- **A measurement artefact of this release.** Impressions and clicks may be counted on different
  bases for these rows. ML-04 already flagged that the whole-lane CTR level (~0.3%) does not
  reconcile with the documented ~2.78%; this band is the extreme of the same puzzle.

Both are async-channel questions. What I will not do is quietly drop the band, or quietly trust it.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

killed = [name for name, v in [("spiky pages have unrepresentative CTR", verdict_2),
                               ("falling impressions means worse CTR", verdict_3)]
          if v in ("FALSE", "OPPOSITE")]

print("FOR A CONTENT TEAM, in three sentences:\n")
print(f"1. Compare a page to its own site before comparing it to the world -- pooled CTR runs from "
      f"{lo:.3f}% to {hi:.3f}%")
print(f"   across the {len(cl)} clients large enough to measure, a {spread:.0f}x spread, so a page can "
      f"look weak against a")
print(f"   global benchmark and be perfectly normal for its own property.")
print(f"\n2. Treat a top-3 page flagged for low CTR as a question, not a job: pages at position 1-2 "
      f"pool to")
print(f"   {band12_pp:.3f}% against {band35_pp:.3f}% for the 3-5 band -- the best-ranked pages "
      f"converting worst --")
print(f"   so until that is explained, a rewrite may be aimed at something a snippet cannot fix.")
print(f"\n3. Read the shape of the month before trusting the month's number: a page whose traffic "
      f"arrived in")
print(f"   one spike, or whose impressions are sliding, is telling you about demand rather than "
      f"about its")
print(f"   snippet -- and those need different medicine.")

print("\n" + "-" * 70)
if killed:
    print(f"Beliefs this month's data did NOT support: {', '.join(killed)}.")
    print("Any rule leaning on them would be prioritising on a hunch. That is the audit paying for "
          "itself.")
else:
    print("No belief was outright falsified this month -- which means the tests were kind, not that "
          "the beliefs are safe. Re-run them on another month before trusting any of them.")
print("Everything above is ONE month, 36 well-instrumented clients, at page-month grain: observed "
      "patterns in a slice, not facts about Google and not causal claims.")

FOR A CONTENT TEAM, in three sentences:

1. Compare a page to its own site before comparing it to the world -- pooled CTR runs from 0.115% to 1.159%
   across the 23 clients large enough to measure, a 10x spread, so a page can look weak against a
   global benchmark and be perfectly normal for its own property.

2. Treat a top-3 page flagged for low CTR as a question, not a job: pages at position 1-2 pool to
   0.118% against 0.394% for the 3-5 band -- the best-ranked pages converting worst --
   so until that is explained, a rewrite may be aimed at something a snippet cannot fix.

3. Read the shape of the month before trusting the month's number: a page whose traffic arrived in
   one spike, or whose impressions are sliding, is telling you about demand rather than about its
   snippet -- and those need different medicine.

----------------------------------------------------------------------
No belief was outright falsified this month -- which means the tests were kind, not that the 

In [22]:
# --- Receipts (the committed artefact) -------------------------------------
import json, os
os.makedirs("work/outputs", exist_ok=True)

receipts = {
    "assignment": "ML-06 - Signal Audit",
    "slice": {"month": MONTH, "eligible_pages": int(len(lane)),
              "clients": int(lane.client_hash_id.nunique())},
    "floors": {"bucket": FLOOR, "cross_cut": CROSS_FLOOR},
    "distributions": {
        "impressions_mean_over_median": round(float(lane.impressions_31d.mean() /
                                                    max(lane.impressions_31d.median(), 1)), 2),
        "pearson_raw": round(float(pear_raw), 3),
        "pearson_log": round(float(pear_log), 3),
        "spearman": round(float(spear), 3),
        "pearson_raw_top1pct_removed": round(float(pear_no_giants), 3),
    },
    "verdicts": {
        "signal_1_normal_ctr_same_across_clients": verdict_1,
        "signal_2_spiky_traffic_distorts_ctr": verdict_2,
        "signal_3_falling_impressions_worse_ctr": verdict_3,
        "flag_linked_position_peers_are_fair": verdict_flag,
    },
    "client_spread": {"lowest_pooled_ctr_pp": float(lo), "highest_pooled_ctr_pp": float(hi),
                      "spread_x": round(float(spread), 1), "clients_over_floor": int(len(cl))},
    "top_band_decomposition": {
        "band_1_2_pages": int(len(band12)),
        "band_1_2_pooled_ctr_pp": round(float(band12_pp), 3),
        "band_3_5_pooled_ctr_pp": round(float(band35_pp), 3),
        "ctr_without_largest_client": round(float(ctr_wo_client), 3),
        "ctr_without_top10pct_pages": round(float(ctr_wo_giants), 3),
        "clients_in_both_bands_over_floor": int(len(both)),
        "cross_cut_floor_used": int(used_floor),
        "clients_worse_at_1_2": (int(worse) if worse is not None else None),
    },
    "claim_discipline": "all verdicts OBSERVED in one month, 36 clients. Decision-support only.",
}
with open("work/outputs/ml06_signal_audit_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2)

print("VERDICTS")
for k, v in receipts["verdicts"].items():
    print(f"  {v:<18} {k}")
print(f"\nclient pooled CTR spread: {lo:.3f}% .. {hi:.3f}% ({spread:.0f}x) across {len(cl)} clients")
print(f"band 1-2 pooled CTR {band12_pp:.3f}% vs band 3-5 {band35_pp:.3f}%")
print("\nsaved -> work/outputs/ml06_signal_audit_receipts.json")

VERDICTS
  FALSE              signal_1_normal_ctr_same_across_clients
  MIXED              signal_2_spiky_traffic_distorts_ctr
  CONFIRMED          signal_3_falling_impressions_worse_ctr
  CONFIRMED          flag_linked_position_peers_are_fair

client pooled CTR spread: 0.115% .. 1.159% (10x) across 23 clients
band 1-2 pooled CTR 0.118% vs band 3-5 0.394%

saved -> work/outputs/ml06_signal_audit_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this audit hands to ML-05

1. **Per-client normalisation is now evidence-backed** rather than an inference from a negative R².
   The model should score a page against its own client's baseline, and I can justify it in one
   sentence with a number in it.
2. **The position 1–2 band is a named open question with a decomposition attached** — one client, a
   few giant pages, or a real property of the top positions. Until it is answered, those picks stay
   lower-confidence in the queue.
3. **Momentum and spikiness describe demand, not snippets.** Whichever way their verdicts fell, they
   belong as filters and caveats on this lane rather than as features pretending to predict it.### What this audit hands to ML-05

1. **Per-client normalisation is now evidence-backed** rather than an inference from a negative R².
   The model should score a page against its own client's baseline, and I can justify it in one
   sentence with a number in it.
2. **The position 1–2 band is a named open question with a decomposition attached** — one client, a
   few giant pages, or a real property of the top positions. Until it is answered, those picks stay
   lower-confidence in the queue.
3. **Momentum and spikiness describe demand, not snippets.** Whichever way their verdicts fell, they
   belong as filters and caveats on this lane rather than as features pretending to predict it.